# Modeling, Production Intake, and Counterfactuals

This notebook is centered on the main business question of the project: what would need to change in a flagged insurance claim so that it would no longer be considered fraudulent? The fraud model is therefore used as the engine behind a counterfactual explanation workflow, not as a black-box scoring demo.

In [1]:
from pathlib import Path
import sys
import json
# import pickle
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from src.config import (
    EVALUATION_PATH, MODEL_METADATA_PATH, MODEL_PATH, 
    PRODUCTION_PACKAGE_RECORD_DIR
)
from src.db.query_from_postgres import fetch_production_decisions
from src.ml.counterfactuals import generate_counterfactual, load_model_and_metadata


## Frozen Development Artifacts

The development model is trained only on the historical PostgreSQL-backed dataset. The purpose of this modeling layer is to support counterfactual generation, analyst understanding, and interpretable fraud review rather than pure score production alone.

In [3]:
model, metadata = load_model_and_metadata()
evaluation = json.loads(EVALUATION_PATH.read_text(encoding='utf-8'))
metadata


{'random_state': 42,
 'test_size': 0.2,
 'validation_size_within_development': 0.2,
 'decision_threshold': 0.83,
 'historical_rows': 54248,
 'development_rows': 43398,
 'validation_rows': 8680,
 'test_rows': 10850,
 'final_training_rows_after_strategy': 80706,
 'historical_positive_rate': 0.070159,
 'historical_date_min': '2018-01-15',
 'historical_date_max': '2024-10-02',
 'suspicious_providers': ['SP-003',
  'SP-013',
  'SP-017',
  'SP-044',
  'SP-058',
  'SP-071',
  'SP-089',
  'SP-098',
  'SP-109',
  'SP-111',
  'SP-115',
  'SP-116'],
 'safe_providers': ['SP-105', 'SP-108', 'SP-067', 'SP-032', 'SP-095'],
 'high_amount_threshold': 28593.39,
 'training_source': 'postgres',
 'historical_source_stage': 'extracted_client_packages_loaded_to_postgres',
 'development_cutoff_date': '2024-10-02',
 'production_start_date': '2024-10-03',
 'production_data_used_in_training': False,
 'classifier': 'xgb',
 'selected_imbalance_strategy': 'oversample',
 'training_accuracy': 0.864075,
 'validation_s

In [4]:
evaluation


{'development_policy': {'production_data_used_in_training': False,
  'production_data_used_in_evaluation': False,
  'notes': 'Pseudo-production claims are reserved for post-development scoring only. Historical evaluation is performed on extracted client-package data loaded from PostgreSQL.'},
 'model_selection_policy': {'objective': 'maximize_precision_subject_to_minimum_recall',
  'minimum_recall': 0.2},
 'historical_holdout': {'roc_auc': 0.764572,
  'precision': 0.52514,
  'recall': 0.247043,
  'f1': 0.336014,
  'decision_threshold': 0.83,
  'deployment_score': 0.450049,
  'top_k': [{'fraction': 0.01,
    'records': 108,
    'precision': 0.777778,
    'recall': 0.110381},
   {'fraction': 0.05,
    'records': 542,
    'precision': 0.431734,
    'recall': 0.30749},
   {'fraction': 0.1,
    'records': 1085,
    'precision': 0.302304,
    'recall': 0.431012}]},
 'deployment_champion_metadata': {'classifier': 'xgb',
  'selected_imbalance_strategy': 'oversample',
  'decision_threshold': 0.

## Post-Development Production Scoring

Recent pseudo-production claims are scored after development is complete. They are not part of the training or evaluation loop.

In [14]:
recent_scores = fetch_production_decisions(limit=500)
recent_scores[['claim_id', 'claim_date', 'claim_amount', 'claim_type', 'service_provider_id', 'fraud_probability', 'model_decision']].sort_values('fraud_probability', ascending=False).head(15)


,claim_id,claim_date,claim_amount,claim_type,service_provider_id,fraud_probability,model_decision
441,CLM-06368-012,2025-10-31,61028.96,bodily_injury,SP-117,0.990991,fraud
114,CLM-02205-013,2025-12-15,71022.15,fire,SP-029,0.985242,fraud
498,CLM-05160-009,2025-10-26,44906.32,bodily_injury,SP-082,0.961103,fraud
362,CLM-03529-007,2025-11-11,25137.04,bodily_injury,SP-111,0.919736,fraud
337,CLM-05534-009,2025-11-15,35352.13,bodily_injury,SP-064,0.906820,fraud
99,CLM-00935-011,2025-12-16,41751.33,bodily_injury,SP-087,0.893520,fraud
401,CLM-03055-002,2025-11-05,27767.55,bodily_injury,SP-058,0.857538,fraud
453,CLM-06332-006,2025-10-30,27238.34,theft,SP-012,0.845098,fraud
180,CLM-07074-004,2025-12-05,25900.81,fire,SP-089,0.844106,fraud
431,CLM-04394-004,2025-11-01,24679.49,bodily_injury,SP-058,0.841951,fraud


Fraud cases:

In [15]:
(
    recent_scores[[
        'claim_id', 'claim_date', 'claim_amount', 'claim_type', 
        'service_provider_id', 'fraud_probability', 'model_decision'
    ]]
    .loc[lambda df: df["model_decision"] == "fraud"]
    .sort_values('fraud_probability', ascending=False)
)


,claim_id,claim_date,claim_amount,claim_type,service_provider_id,fraud_probability,model_decision
441,CLM-06368-012,2025-10-31,61028.96,bodily_injury,SP-117,0.990991,fraud
114,CLM-02205-013,2025-12-15,71022.15,fire,SP-029,0.985242,fraud
498,CLM-05160-009,2025-10-26,44906.32,bodily_injury,SP-082,0.961103,fraud
362,CLM-03529-007,2025-11-11,25137.04,bodily_injury,SP-111,0.919736,fraud
337,CLM-05534-009,2025-11-15,35352.13,bodily_injury,SP-064,0.906820,fraud
99,CLM-00935-011,2025-12-16,41751.33,bodily_injury,SP-087,0.893520,fraud
401,CLM-03055-002,2025-11-05,27767.55,bodily_injury,SP-058,0.857538,fraud
453,CLM-06332-006,2025-10-30,27238.34,theft,SP-012,0.845098,fraud
180,CLM-07074-004,2025-12-05,25900.81,fire,SP-089,0.844106,fraud
431,CLM-04394-004,2025-11-01,24679.49,bodily_injury,SP-058,0.841951,fraud


## Package Intake Records

The API can receive production claim packages composed of JSON plus attachments. Those packages are stored locally, the structured claim is rebuilt through local rule-based document extraction, and the final production decision is also persisted to PostgreSQL for SQL-based follow-up.

In [16]:
package_records = sorted(PRODUCTION_PACKAGE_RECORD_DIR.glob('*.json'))
len(package_records), package_records[:5]


(3,
 [WindowsPath('C:/Users/aurel/Desktop/Projects/Insurance Claim Counterfactual Simulator/data/processed/production_package_records/CLM-00001-007.json'),
  WindowsPath('C:/Users/aurel/Desktop/Projects/Insurance Claim Counterfactual Simulator/data/processed/production_package_records/CLM-00028-018.json'),
  WindowsPath('C:/Users/aurel/Desktop/Projects/Insurance Claim Counterfactual Simulator/data/processed/production_package_records/CLM-06485-001.json')])

In [17]:
if package_records:
    sample_record = json.loads(package_records[0].read_text(encoding='utf-8'))
    sample_record
else:
    {'message': 'No production package records stored yet. Use POST /intake/package or POST /predict/package first.'}


## Counterfactual Example

Counterfactual search remains heuristic. It looks for plausible changes such as lowering the claim amount, increasing the gap since the previous claim, changing provider, or increasing policy age.

In [18]:
candidate_df = recent_scores.sort_values('fraud_probability', ascending=False).head(25)
counterfactual_result = None
for _, row in candidate_df.iterrows():
    payload = row[['claim_id', 'customer_id', 'claim_date', 'policy_start_date', 'policy_age_days', 'claim_amount', 'claim_type', 'customer_age', 'num_previous_claims', 'time_since_last_claim_days', 'service_provider_id', 'location', 'weather_condition']].to_dict()
    result = generate_counterfactual(payload, model=model, metadata=metadata)
    if result['status'] == 'found':
        counterfactual_result = {'claim_id': row['claim_id'], 'result': result}
        break
counterfactual_result


{'claim_id': 'CLM-02205-013',
 'result': {'status': 'found',
  'heuristic': True,
  'original_probability': 0.985242,
  'counterfactual_probability': 0.787997,
  'decision_threshold': 0.83,
  'changes': {'original_values': {'claim_amount': Decimal('71022.15'),
    'service_provider_id': 'SP-029'},
   'proposed_values': {'claim_amount': 56817.72,
    'service_provider_id': 'SP-105'}}}}

## Interpretation

The core value of this project is not just that it produces a fraud probability. Its real value is that it helps analysts and insurers understand what is making a claim look suspicious and what plausible changes would move the claim below the alert threshold. That makes the workflow more useful for human review, model governance, and operational decision support.